# Regime Model 1st try

In [6]:
import numpy as np
import pandas as pd
from hmmlearn.hmm import GaussianHMM
import logging
import warnings
import feature_selection as fs
from plotly.subplots import make_subplots
import plotly.graph_objects as go

warnings.filterwarnings('ignore')

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

class RegimeSwitchModel:
    def __init__(self, symbol, interval='1d', train_pct=0.8, strategy='long-only', confidence_threshold=0.75):
        self.symbol = symbol
        self.interval = interval
        self.train_pct = train_pct
        if strategy not in ['long-only', 'long-short']:
            raise ValueError("Strategy must be either 'long-only' or 'long-short'")
        self.strategy = strategy
        self.confidence_threshold = confidence_threshold  # Minimum confidence for trade execution
        if not 0 <= confidence_threshold <= 1:
            raise ValueError("confidence_threshold must be between 0 and 1")
        self.data = None
        self.features = None
        self.selected_features = None
        self.train_data = None
        self.state_probabilities = None  # Store confidence levels
        
        # Define paths for data
        self.onchain_data_path = "/Users/valter.rebelo/MissionControl/data/onchainData"
        self.macro_data_path = "/Users/valter.rebelo/MissionControl/data/macro/fredData"
        
        # Define macro features to use
        self.macro_features = [
            'treasury5YInflationExpectation', 
            'treasury5YInflationForwardRate', 
            'creditSpreads', 
            'vix', 
            'sp500', 
            'globalCbLiquidity'
        ]

    def load_data(self):
        """
        Load price data from files and prepare basic dataframe
        """
        try:
            # Load price data
            df_1 = pd.read_csv(f"/Users/valter.rebelo/MissionControl/data/micro/candleData/{self.symbol}_candles.csv")
            df_1['date'] = pd.to_datetime(df_1['date'])
            df_1.set_index('date', inplace=True)

            # Load market cap data
            df_2 = pd.read_csv(f"/Users/valter.rebelo/MissionControl/data/micro/assetData/{self.symbol}.csv")
            df_2['date'] = pd.to_datetime(df_2['date'])
            df_2.set_index('date', inplace=True)

            # Load BTC data for non-BTC assets
            btc_df = pd.read_csv("/Users/valter.rebelo/MissionControl/data/micro/candleData/bitcoin_candles.csv")
            btc_df['date'] = pd.to_datetime(btc_df['date'])
            btc_df.set_index('date', inplace=True)

            # Merge price and market cap data
            data = pd.merge(df_1, df_2[['total_volume', 'market_cap']], on='date', how='inner')
            data.rename(columns={'total_volume': 'Volume', 'open': 'Open', 'high': 'High', 'low': 'Low', 'close': 'Close'}, inplace=True)
            data.index.name = 'Date'

            # Add BTC relative price for non-BTC assets
            if self.symbol != "bitcoin":
                data['close_btc'] = (data['Close'] / btc_df['close']) * 100
                data.dropna(inplace=True)

            # Filter data for bitcoin
            if self.symbol == "bitcoin":
                data = data[data.index >= '2020-01-01']
            
            # Store the basic data
            self.data = data
            logging.info(f"Loaded {len(data)} rows of basic data for {self.symbol}")
            
            return data
            
        except Exception as e:
            logging.error(f"Data loading failed for {self.symbol}: {str(e)}")
            raise

    def generate_all_features(self):
        """
        Generate all features using feature_selection module with NO feature computation in notebook
        """
        if self.data is None:
            raise ValueError("Basic data must be loaded first. Call load_data() before generate_all_features().")
        
        try:
            # Call the feature generation function from feature_selection.py
            feature_results = fs.generate_features(
                data=self.data,
                symbol=self.symbol,
                include_technical=True,
                include_onchain=True,
                include_macro=True,
                onchain_data_path=self.onchain_data_path,
                macro_data_path=self.macro_data_path,
                macro_features=self.macro_features,
                verbose=True,
                show_plots=True  # Set to False to reduce logging noise
            )
            # Extract the processed data from the returned dictionary
            self.features = feature_results['processed_data']
            
            logging.info(f"Generated {len(self.features.columns)} features using feature_selection module")
            return self.features
        
        except Exception as e:
            logging.error(f"Feature generation failed: {str(e)}")
            raise

    def select_features(self, correlation_threshold=0.15, min_consensus=2, verbose=True):
        """
        Select the best features using feature selection from feature_selection module
        
        Args:
            correlation_threshold: Minimum absolute correlation with target
            min_consensus: Minimum number of methods that must select a feature
            verbose: Whether to print detailed information during selection
            
        Returns:
            List of selected feature names
        """
        if self.features is None:
            raise ValueError("Features must be generated first. Call generate_all_features() before select_features().")
        
        # Use the existing select_features function from the module
        results = fs.select_features(
            data=self.features,
            correlation_threshold=correlation_threshold,
            min_consensus=min_consensus,
            verbose=verbose
        )
        
        # Extract the recommended features from the results
        final_features = results['consensus_features']
        
        logging.info(f"Selected {len(final_features)} features using feature_selection module")
        
        # Validate we have enough features
        if len(final_features) < 3:
            logging.warning(f"Feature selection returned too few features: {final_features}. Using default features.")
            # Fallback to some default features if selection fails
            final_features = ['rsi_14']
        
        self.selected_features = final_features
        
        # Make sure we have log_close for modeling
        if 'log_close' not in self.features.columns:
            self.features['log_close'] = np.log(self.features['Close'])
        
        # Create model_data with log_return, log_close, and selected features
        self.model_data = self.features[['log_return', 'log_close', 'Open', 'Close', 'Volume'] + self.selected_features]
        
        return self.selected_features

    def set_manual_features(self, feature_list):
        """
        Manually set the features to use for modeling, overriding automatic feature selection.
        
        Args:
            feature_list: List of feature names to use
            
        Returns:
            List of validated feature names that exist in the data
        """
        if self.features is None:
            raise ValueError("Features must be generated first. Call generate_all_features() before set_manual_features().")
        
        # Validate that the features exist in the data
        available_features = self.features.columns.tolist()
        valid_features = [f for f in feature_list if f in available_features]
        
        if len(valid_features) == 0:
            raise ValueError("None of the specified features exist in the data.")
        
        if len(valid_features) < len(feature_list):
            missing = set(feature_list) - set(valid_features)
            logging.warning(f"Some requested features are not available: {missing}")
        
        # Set the selected features
        self.selected_features = valid_features
        logging.info(f"Manually set {len(valid_features)} features: {valid_features}")
        
        # Make sure we have log_close for modeling
        if 'log_close' not in self.features.columns:
            self.features['log_close'] = np.log(self.features['Close'])
        
        # Create model_data with essential price columns, log_return, log_close, and selected features
        essential_columns = ['Open', 'High', 'Low', 'Close', 'Volume']
        essential_available = [col for col in essential_columns if col in self.features.columns]
        
        columns_to_include = essential_available + ['log_return', 'log_close'] + self.selected_features
        columns_to_include = list(dict.fromkeys(columns_to_include))
        
        self.model_data = self.features[columns_to_include]
        
        return valid_features

    def split_data(self, data, embargo_percent=0.01):
        try:
            total_rows = len(data)
            train_end_idx = int(total_rows * self.train_pct)
            embargo_end_idx = int(train_end_idx + (total_rows * embargo_percent))
            
            train = data.iloc[:train_end_idx]
            embargo = data.iloc[train_end_idx:embargo_end_idx]
            test = data.iloc[embargo_end_idx:]
            
            self.train_data = train

            if len(train) < 50 or len(test) < 50:
                raise ValueError("Train or test set too small (<50 rows).")
            logging.info(f"Split data: train={len(train)}, embargo={len(embargo)}, test={len(test)}")
            return train, embargo, test
        
        except Exception as e:
            logging.error(f"Data splitting failed: {str(e)}")
            raise

    def normalize_features(self, train, test, features):
        train_mean = train[features].mean()
        train_std = train[features].std()
        train_normalized = (train[features] - train_mean) / train_std
        test_normalized = (test[features] - train_mean) / train_std

        return (pd.DataFrame(train_normalized, index=train.index, columns=features),
                pd.DataFrame(test_normalized, index=test.index, columns=features))

    def train_hmm(self, train, features):
        """
        Train a Hidden Markov Model (HMM) on the provided training data.
        
        Args:
            train: Training data DataFrame
            features: List of features to use for training
            
        Returns:
            Trained HMM model
        """
        try:
            if train is None:
                raise ValueError("Training data is None. Please run load_data, generate_all_features, select_features, and split_data first.")
            
            f_train_normalized, _ = self.normalize_features(train, train, features)
            hmm = GaussianHMM(n_components=3, covariance_type='full', n_iter=5000, random_state=42)
            hmm.fit(f_train_normalized)
            if not hmm.monitor_.converged:
                logging.warning("HMM training did not converge.")
            logging.info("HMM trained successfully")
            return hmm
        except Exception as e:
            logging.error(f"HMM training failed: {str(e)}")
            raise

    def predict_states(self, hmm, f_test, optimal_states, test_data):
        """
        Predict states using a single HMM, map to trading actions, and filter by confidence.
        
        Args:
            hmm: Trained HMM model
            f_test: Normalized test features
            optimal_states: Dictionary mapping state indices to trading actions
            test_data: Original test data for alignment
        
        Returns:
            Series of predicted states
        """
        # Predict states and probabilities
        hidden_states = hmm.predict(f_test)
        state_probs = hmm.predict_proba(f_test)
        predicted_states = [optimal_states.get(s, 'Flat') for s in hidden_states]
        confidences = [state_probs[i, state] for i, state in enumerate(hidden_states)]

        # Debugging logs
        logging.info(f"Length of f_test: {len(f_test)}")
        logging.info(f"Length of test_data: {len(test_data)}")
        logging.info(f"Length of test_data.index: {len(test_data.index)}")
        logging.info(f"Length of predicted_states: {len(predicted_states)}")
        logging.info(f"Test data index range: {test_data.index[0]} to {test_data.index[-1]}")

        # Align the index starting from the second row to match pct_change in simulate_trading
        if len(predicted_states) == len(test_data):
            # Filter states based on confidence threshold
            filtered_states = []
            for state, conf in zip(predicted_states[1:], confidences[1:]):
                if state in ['Long', 'Short'] and conf < self.confidence_threshold:
                    filtered_states.append('Flat')
                else:
                    filtered_states.append(state)

            state_series = pd.Series(filtered_states, index=test_data.index[1:])
            confidence_series = pd.Series(confidences[1:], index=test_data.index[1:])
        else:
            logging.error(f"Mismatch: predicted_states length ({len(predicted_states)}) does not match test_data length ({len(test_data)})")
            raise ValueError(f"Length mismatch: {len(predicted_states)} predictions vs {len(test_data)} test rows")

        # Store state probabilities
        self.state_probabilities = pd.DataFrame({
            'confidence': confidence_series,
            'state': state_series
        })

        return state_series

    def find_optimal_states(self, hmm, train_data, features):
        """
        Find optimal state mappings that maximize Sharpe ratio on training data
        
        Args:
            hmm: Trained HMM model
            train_data: Training data
            features: Features used for training
            
        Returns:
            Dictionary mapping state indices to trading actions ('Long', 'Short', 'Flat')
        """
        try:
            f_train_normalized, _ = self.normalize_features(train_data, train_data, features)
            hidden_states = hmm.predict(f_train_normalized)
            unique_states = np.unique(hidden_states)
            
            state_returns = {}
            for state in unique_states:
                state_mask = (hidden_states == state)
                if sum(state_mask) > 0:
                    if 'log_return' in train_data.columns:
                        returns = train_data.loc[state_mask, 'log_return'].values
                    else:
                        returns = np.diff(np.log(train_data.loc[state_mask, 'Close'].values))
                        
                    state_returns[state] = {
                        'mean': np.mean(returns),
                        'std': np.std(returns) if len(returns) > 1 else 1e-6,
                        'sharpe': np.mean(returns) / (np.std(returns) if len(returns) > 1 else 1e-6),
                        'count': len(returns)
                    }
            
            optimal_states = {}
            sorted_states = sorted(state_returns.items(), key=lambda x: x[1]['sharpe'], reverse=True)
            
            if self.strategy == 'long-only':
                for i, (state, _) in enumerate(sorted_states):
                    if i == 0 and state_returns[state]['mean'] > 0:
                        optimal_states[state] = 'Long'
                    else:
                        optimal_states[state] = 'Flat'
            else:
                for i, (state, metrics) in enumerate(sorted_states):
                    if i == 0 and metrics['mean'] > 0:
                        optimal_states[state] = 'Long'
                    elif i == len(sorted_states) - 1 and metrics['mean'] < 0:
                        optimal_states[state] = 'Short'
                    else:
                        optimal_states[state] = 'Flat'
            
            logging.info(f"Optimal state mappings based on Sharpe ratio: {optimal_states}")
            for state, action in optimal_states.items():
                if state in state_returns:
                    metrics = state_returns[state]
                    logging.info(f"State {state} -> {action}: Sharpe={metrics['sharpe']:.2f}, Mean={metrics['mean']:.4f}, Count={metrics['count']}")
            
            return optimal_states
        except Exception as e:
            logging.error(f"Optimal state finding failed: {str(e)}")
            state_means = hmm.means_[:, 0]
            default_mapping = {}
            for state in range(len(state_means)):
                if state == np.argmax(state_means):
                    default_mapping[state] = 'Long'
                elif self.strategy == 'long-short' and state == np.argmin(state_means):
                    default_mapping[state] = 'Short'
                else:
                    default_mapping[state] = 'Flat'
            logging.warning(f"Using fallback state mapping: {default_mapping}")
            return default_mapping

    def ensemble_predict(self, test, features):
        """
        Train a single HMM and predict states.
        
        Args:
            test: Test data
            features: Features to use for prediction
        
        Returns:
            Series of predicted states
        """
        try:
            # Train a single HMM on daily data
            hmm = self.train_hmm(self.train_data, features)
            optimal_states = self.find_optimal_states(hmm, self.train_data, features)
            
            # Normalize test features
            f_train_normalized, f_test_normalized = self.normalize_features(self.train_data, test, features)
            
            # Log shapes for debugging
            logging.info(f"Train data shape: {self.train_data.shape}, Test data shape: {test.shape}")
            logging.info(f"f_train_normalized shape: {f_train_normalized.shape}, f_test_normalized shape: {f_test_normalized.shape}")
            logging.info(f"Test data index: {test.index[0]} to {test.index[-1]}")

            # Predict states for test data
            states = self.predict_states(hmm, f_test_normalized, optimal_states, test)
            logging.info("Prediction completed")
            return states
        except Exception as e:
            logging.error(f"Prediction failed: {str(e)}")
            raise

    def simulate_trading(self, test, states, initial_capital=1000):
        try:
            # Strategy returns
            shifted_states = states.shift(1).fillna('Flat')
            position_multiplier = (shifted_states == 'Long').astype(int) - (shifted_states == 'Short').astype(int)
            
            # Calculate raw returns
            raw_returns = test['Open'].pct_change()
            
            # Calculate trading costs (0.01% per trade)
            trading_cost = 0.01  # 0.01% expressed as decimal
            # Detect position changes (which trigger costs)
            position_changes = position_multiplier.diff().fillna(0).abs()
            # Apply costs when positions change
            costs = position_changes * trading_cost
            
            # Apply position and subtract costs
            returns = (raw_returns * position_multiplier) - costs
            cum_returns = (1 + returns).cumprod()
            
            # Calculate monetary values
            portfolio_value = initial_capital * cum_returns
            cumulative_costs = initial_capital * (1 - (1 - costs).cumprod())
            
            # Buy and hold returns (no trading costs as it's buy once and hold)
            bnh_returns = test['Close'].pct_change().fillna(0.0)
            bnh_cum_returns = (1 + bnh_returns).cumprod()
            bnh_portfolio_value = initial_capital * bnh_cum_returns
            
            result = pd.DataFrame({
                'Close': test['Close'], 
                'state': states,
                'returns': returns, 
                'cum_returns': cum_returns,
                'portfolio_value': portfolio_value,
                'cumulative_costs': cumulative_costs,
                'bnh_returns': bnh_returns,
                'bnh_cum_returns': bnh_cum_returns,
                'bnh_portfolio_value': bnh_portfolio_value
            })
            logging.info("Trading simulation completed")
            logging.info(f"Final portfolio value: ${portfolio_value.iloc[-1]:.2f}, Total trading costs: ${cumulative_costs.iloc[-1]:.2f}")
            
            # Automatically plot the backtest results
            self.plot_backtest(result, initial_capital)
            
            return result
        except Exception as e:
            logging.error(f"Trading simulation failed: {str(e)}")
            raise

    def plot_backtest(self, result, initial_capital=1000):
        """Plot backtest results with trading costs and actual confidence levels"""
        try:
            # Create figure with four subplots for better separation
            fig = make_subplots(rows=4, cols=1, 
                              shared_xaxes=True, 
                              vertical_spacing=0.1,  # Increased spacing for better separation
                              subplot_titles=('Portfolio Performance', 'Trading State', 'Predicted State Confidence', 'Cumulative Trading Costs'),
                              row_heights=[0.4, 0.25, 0.25, 0.1])  # Adjusted heights for balanced separation

            # Add portfolio value
            fig.add_trace(go.Scatter(x=result.index, y=result['portfolio_value'], 
                                   mode='lines', name='Strategy', line=dict(color='blue', width=1)),
                        row=1, col=1)
            fig.add_trace(go.Scatter(x=result.index, y=result['bnh_portfolio_value'], 
                                   mode='lines', name='Buy & Hold', line=dict(color='gray', width=1)),
                        row=1, col=1)
            fig.add_trace(go.Scatter(x=[result.index[0], result.index[-1]], 
                                   y=[initial_capital, initial_capital],
                                   mode='lines', name='Initial Capital', 
                                   line=dict(color='black', dash='dash', width=0.8)),
                        row=1, col=1)

            # Add trading states
            state_numeric = result['state'].map({'Long': 1, 'Flat': 0, 'Short': -1})
            fig.add_trace(go.Scatter(x=result.index, y=state_numeric, mode='lines', 
                                   name='Trading State', line=dict(color='green', width=1)),
                        row=2, col=1)

            # Add actual confidence levels from the model
            if self.state_probabilities is not None and 'confidence' in self.state_probabilities.columns:
                fig.add_trace(go.Scatter(x=self.state_probabilities.index, 
                                       y=self.state_probabilities['confidence'],
                                       mode='lines', name='Predicted Confidence',
                                       line=dict(color='purple', width=1)),
                            row=3, col=1)
                # Add a horizontal line for the confidence threshold
                fig.add_trace(go.Scatter(x=[self.state_probabilities.index[0], self.state_probabilities.index[-1]], 
                                       y=[self.confidence_threshold, self.confidence_threshold], 
                                       mode='lines', name='Threshold',
                                       line=dict(color="red", width=0.8, dash="dash")),
                            row=3, col=1)
            else:
                logging.warning("State probabilities not available for plotting confidence.")

            # Add cumulative trading costs
            fig.add_trace(go.Scatter(x=result.index, y=result['cumulative_costs'], 
                                   mode='lines', name='Trading Costs', 
                                   line=dict(color='red', width=1)),
                        row=4, col=1)

            # Update layout with elegant, minimalist design
            fig.update_layout(height=1100,  # Increased height to accommodate separation
                            width=1000,
                            showlegend=True,
                            legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
                            plot_bgcolor='white',
                            paper_bgcolor='white',
                            margin=dict(t=120, b=60, l=50, r=50))  # Adjusted margins for balance

            # Update axes with light gray grids and titles
            fig.update_yaxes(title_text="Portfolio Value ($)", row=1, col=1, gridcolor='lightgray', zeroline=False, showline=True, linewidth=0.5, linecolor='lightgray')
            fig.update_yaxes(title_text="State", tickvals=[-1, 0, 1], ticktext=['Short', 'Flat', 'Long'], row=2, col=1, gridcolor='lightgray', zeroline=False, showline=True, linewidth=0.5, linecolor='lightgray')
            fig.update_yaxes(title_text="Confidence", range=[0, 1], row=3, col=1, gridcolor='lightgray', zeroline=False, showline=True, linewidth=0.5, linecolor='lightgray')
            fig.update_yaxes(title_text="Trading Costs ($)", row=4, col=1, gridcolor='lightgray', zeroline=False, showline=True, linewidth=0.5, linecolor='lightgray')
            fig.update_xaxes(title_text="Date", row=4, col=1, gridcolor='lightgray', showline=True, linewidth=0.5, linecolor='lightgray')

            # Display the figure
            fig.show()
            
            return fig
        except Exception as e:
            logging.error(f"Plotting backtest failed: {str(e)}")
            raise

    def evaluate(self, results):
        try:
            # Strategy evaluation
            logging.info(f"state sample: {results['state'].head().tolist()}")
            cum_returns = results['cum_returns'].fillna(1.0)
            logging.info(f"Final cum_returns: {cum_returns.iloc[-1]}, length: {len(results)}")
            returns = results['returns'].fillna(0.0)
            
            # Calculate strategy metrics
            ann_ret = (cum_returns.iloc[-1] ** (365/len(results))) - 1
            sharpe = (returns.mean() / returns.std()) * np.sqrt(365)
            # Calculate Sortino ratio (using negative returns only for denominator)
            neg_returns = returns[returns < 0]
            sortino = (returns.mean() / neg_returns.std()) * np.sqrt(365) if len(neg_returns) > 0 else np.inf
            ann_vol = returns.std() * np.sqrt(365)
            drawdowns = cum_returns / cum_returns.cummax() - 1
            max_dd = drawdowns.min()
            
            # Count strategy switches
            state = results['state']
            switches = sum(state.iloc[i] != state.iloc[i-1] for i in range(1, len(state)))
            
            # Calculate total trading costs
            if 'cumulative_costs' in results.columns:
                total_costs = results['cumulative_costs'].iloc[-1]
            else:
                total_costs = "N/A"
            
            # Buy and hold metrics
            bnh_returns = results['bnh_returns']
            bnh_cum_returns = results['bnh_cum_returns']
            bnh_ann_ret = (bnh_cum_returns.iloc[-1] ** (365/len(results))) - 1
            bnh_sharpe = (bnh_returns.mean() / bnh_returns.std()) * np.sqrt(365)
            # Calculate benchmark Sortino
            bnh_neg_returns = bnh_returns[bnh_returns < 0]
            bnh_sortino = (bnh_returns.mean() / bnh_neg_returns.std()) * np.sqrt(365) if len(bnh_neg_returns) > 0 else np.inf
            bnh_ann_vol = bnh_returns.std() * np.sqrt(365)
            bnh_drawdowns = bnh_cum_returns / bnh_cum_returns.cummax() - 1
            bnh_max_dd = bnh_drawdowns.min()
            
            # Create metrics table
            metrics_df = pd.DataFrame({
                'Metric': ['Annualized Return', 'Sharpe Ratio', 'Sortino Ratio', 'Annualized Volatility', 
                          'Maximum Drawdown', 'Final Cum Return', 'Switches', 'Total Trading Costs'],
                f'HMM {self.strategy} Strategy': [ann_ret, sharpe, sortino, ann_vol, max_dd, cum_returns.iloc[-1], switches, total_costs],
                'Buy & Hold': [bnh_ann_ret, bnh_sharpe, bnh_sortino, bnh_ann_vol, bnh_max_dd, 
                              bnh_cum_returns.iloc[-1], 'N/A', 'N/A']
            })
            metrics_df.set_index('Metric', inplace=True)
            logging.info(f"Evaluation metrics:\n{metrics_df}")
            return metrics_df
        except Exception as e:
            logging.error(f"Evaluation failed: {str(e)}")
            raise

In [11]:
# Load data and generate features as usual
model = RegimeSwitchModel('bitcoin', strategy='long-only', train_pct=0.8, confidence_threshold=0.9)
data = model.load_data()
features = model.generate_all_features()


2025-03-07 19:27:16,354 - INFO - Loaded 1890 rows of basic data for bitcoin
2025-03-07 19:27:16,355 - INFO - Starting feature generation pipeline...
2025-03-07 19:27:16,363 - INFO - Generating technical indicators...
2025-03-07 19:27:16,389 - INFO - Loading and processing on-chain metrics...
2025-03-07 19:27:16,389 - INFO - Loading on-chain data for bitcoin...


2025-03-07 19:27:16,551 - INFO - Loaded 28 on-chain metrics
2025-03-07 19:27:16,579 - INFO - Processed 28 on-chain metrics out of 28 total
2025-03-07 19:27:16,579 - INFO - Loading and processing macro features...
2025-03-07 19:27:16,579 - INFO - Loading macro data...
2025-03-07 19:27:16,604 - INFO - Loaded 6 macro metrics
2025-03-07 19:27:16,613 - WARNING - Column treasury5YInflationExpectation has too many zero standard deviation values, skipping z-score calculation
2025-03-07 19:27:16,616 - WARNING - Column treasury5YInflationForwardRate has too many zero standard deviation values, skipping z-score calculation
2025-03-07 19:27:16,619 - WARNING - Column creditSpreads has too many zero standard deviation values, skipping z-score calculation
2025-03-07 19:27:16,622 - WARNING - Column vix has too many zero standard deviation values, skipping z-score calculation
2025-03-07 19:27:16,624 - WARNING - Column sp500 has too many zero standard deviation values, skipping z-score calculation
2025-

2025-03-07 19:27:17,014 - INFO - Generated 318 features using feature_selection module


In [12]:
selected_features = model.select_features(correlation_threshold=0.2, min_consensus=3)
selected_features



STARTING CONSENSUS FEATURE SELECTION
Input data shape: (1890, 318) (318 features)
Target column: log_return
Correlation threshold: 0.2
Minimum consensus required: 3 methods
--------------------------------------------------------------------------------

STEP 1: Correlation Filtering
--------------------------------------------------
Calculating correlations with log_return...
Range of correlations: -0.3281 to 0.6275
Mean absolute correlation: 0.0878
Features with positive correlation >= 0.2: 45
Features with negative correlation <= -0.2: 4

Top 5 positively correlated features:
  price_sma7_ratio: +0.6275
  price_equilibrium_zscore: +0.5872
  rsi_7: +0.4481
  price_equilibrium: +0.4254
  stoch_k: +0.4165

Top 5 negatively correlated features:
  relative_unrealized_loss_zscore: -0.3281
  utxo_loss_count_zscore: -0.3242
  relative_unrealized_loss_mom7: -0.2727
  utxo_loss_count_mom7: -0.2470

Features after correlation filtering (|correlation| >= 0.2): 49
Removed 268 features with low 


Complete list of consensus features:


2025-03-07 19:27:26,024 - INFO - Selected 12 features using feature_selection module



Feature Selection Summary:
                                   correlation  consensus_count  lasso  ridge  \
price_sma7_ratio                      0.627545                3   True   True   
price_equilibrium_zscore              0.587180                3   True   True   
sth_sopr_zscore                       0.415133                3   True   True   
sth_sopr                              0.381519                3   True   True   
resistance_relative                   0.353378                3   True   True   
rsi_14                                0.343207                3   True   True   
entity_adj_nupl_mom7                  0.335802                3   True   True   
relative_unrealized_loss_zscore       0.328058                3   True   True   
price_sma30_ratio                     0.311939                3   True   True   
mvrv_lth_mom7                         0.280197                3   True   True   
ssr_oscillator                        0.251630                3   True   True   


['ssr_oscillator',
 'entity_adj_nupl_mom7',
 'rsi_14',
 'ssr_oscillator_mom7',
 'mvrv_lth_mom7',
 'sth_sopr',
 'resistance_relative',
 'price_equilibrium_zscore',
 'sth_sopr_zscore',
 'relative_unrealized_loss_zscore',
 'price_sma30_ratio',
 'price_sma7_ratio']

In [13]:
manual_features = [#'relative_unrealized_profit_mom7',
 'rsi_14',
 #'pct_supply_in_profit_mom7',
 'mvrv_mom7',
 'price_sma30_ratio',
 'pct_supply_in_profit_zscore',
 'entity_adj_nupl_mom7',
 #'price_sma7_ratio',
 'relative_unrealized_loss_zscore',
 'sth_sopr_zscore',
 'price_drawdown_relative',
# 'price_equilibrium_zscore',
 #### New Features ###
 

]

['relative_unrealized_profit_mom7',
 'rsi_14',
 'pct_supply_in_profit_mom7',
 'mvrv_mom7', # deixa dar uma respirada
 'price_sma30_ratio', # captura bons pontos de entrada
 'pct_supply_in_profit_zscore', # captura bons pontos de saída 
 'entity_adj_nupl_mom7',
 'price_sma7_ratio',
 'relative_unrealized_loss_zscore',
 'sth_sopr_zscore',
 'price_drawdown_relative',
]


selected_features = model.set_manual_features(manual_features)

# Continue with the rest of your workflow as usual


2025-03-07 19:27:26,034 - INFO - Manually set 8 features: ['rsi_14', 'mvrv_mom7', 'price_sma30_ratio', 'pct_supply_in_profit_zscore', 'entity_adj_nupl_mom7', 'relative_unrealized_loss_zscore', 'sth_sopr_zscore', 'price_drawdown_relative']


In [14]:
train, embargo, test = model.split_data(model.model_data, embargo_percent=0.05)
print(f"Train set: {len(train)} rows")
print(f"Embargo set: {len(embargo)} rows")
print(f"Test set: {len(test)} rows")

2025-03-07 19:27:26,040 - INFO - Split data: train=1512, embargo=94, test=284


Train set: 1512 rows
Embargo set: 94 rows
Test set: 284 rows


In [15]:

states = model.ensemble_predict(test, selected_features)
results = model.simulate_trading(test, states)
metrics = model.evaluate(results)



2025-03-07 19:27:26,504 - INFO - HMM trained successfully
2025-03-07 19:27:26,509 - INFO - Optimal state mappings based on Sharpe ratio: {1: 'Long', 0: 'Flat', 2: 'Flat'}
2025-03-07 19:27:26,510 - INFO - State 1 -> Long: Sharpe=0.36, Mean=0.0119, Count=459
2025-03-07 19:27:26,510 - INFO - State 0 -> Flat: Sharpe=0.06, Mean=0.0012, Count=699
2025-03-07 19:27:26,511 - INFO - State 2 -> Flat: Sharpe=-0.25, Mean=-0.0123, Count=354
2025-03-07 19:27:26,514 - INFO - Train data shape: (1512, 15), Test data shape: (284, 15)
2025-03-07 19:27:26,514 - INFO - f_train_normalized shape: (1512, 8), f_test_normalized shape: (284, 8)
2025-03-07 19:27:26,514 - INFO - Test data index: 2024-05-25 00:00:00 to 2025-03-04 00:00:00
2025-03-07 19:27:26,521 - INFO - Length of f_test: 284
2025-03-07 19:27:26,522 - INFO - Length of test_data: 284
2025-03-07 19:27:26,522 - INFO - Length of test_data.index: 284
2025-03-07 19:27:26,523 - INFO - Length of predicted_states: 284
2025-03-07 19:27:26,523 - INFO - Test da

2025-03-07 19:27:26,595 - INFO - state sample: [nan, 'Long', 'Long', 'Long', 'Long']
2025-03-07 19:27:26,595 - INFO - Final cum_returns: 2.1251123884821586, length: 284
2025-03-07 19:27:26,598 - INFO - Evaluation metrics:
                       HMM long-only Strategy Buy & Hold
Metric                                                  
Annualized Return                    1.634844   0.341139
Sharpe Ratio                         3.024159    0.84091
Sortino Ratio                        4.721727   1.346387
Annualized Volatility                0.339424    0.49073
Maximum Drawdown                    -0.104614  -0.242495
Final Cum Return                     2.125112   1.256566
Switches                            18.000000        N/A
Total Trading Costs                165.486239        N/A
